# 03 · Deploy an open-source model on a free T4

**Runtime → Change runtime type → T4 GPU**, then run top to bottom (~10 min).

Serves `unsloth/Llama-3.2-1B-Instruct` with vLLM behind an OpenAI-compatible API, load-tests it, reads `/metrics`, and writes the results table for the README.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Your copy of this repo (push it to GitHub first), or upload the folder to Colab instead
REPO = "https://github.com/pribinkbenny/03-deploy.git"
!git clone -q $REPO project || echo 'clone failed: upload the 03-deploy folder and rename it to project'
%cd project
!pip install -q vllm httpx matplotlib

## 1. Start the server
T4 has no bf16, so `--dtype half`. `--max-model-len 4096` keeps the KV cache small enough for 15 GB.

In [ ]:
MODEL = "unsloth/Llama-3.2-1B-Instruct"
!nohup vllm serve $MODEL --dtype half --max-model-len 4096 --gpu-memory-utilization 0.90 > vllm.log 2>&1 &

In [ ]:
import time, httpx
for i in range(180):
    try:
        if httpx.get('http://localhost:8000/health', timeout=2).status_code == 200:
            print(f'server up after {i*5}s'); break
    except Exception:
        pass
    time.sleep(5)
else:
    print(open('vllm.log').read()[-3000:])

## 2. Working endpoint ✓

In [ ]:
!curl -s localhost:8000/v1/chat/completions -H 'Content-Type: application/json' \
  -d '{"model": "unsloth/Llama-3.2-1B-Instruct", "messages": [{"role": "user", "content": "Say hello in 5 words."}], "max_tokens": 20}' | python -m json.tool

In [ ]:
!curl -s localhost:8000/metrics | grep -E '^vllm:(time_to_first_token_seconds|inter_token_latency_seconds|time_per_output_token_seconds)_(sum|count)'

## 3. Load test ✓
32 streaming requests at each concurrency level. Watch TTFT rise and throughput climb, then flatten as the GPU saturates.

In [ ]:
!python -m deploy.loadtest --base-url http://localhost:8000/v1 --model $MODEL \
   --concurrency 1 4 16 32 --requests 64 --max-tokens 128 \
   --label "Colab T4 · vLLM · fp16" --out results/loadtest.json

In [ ]:
!python -m deploy.metrics --url http://localhost:8000/metrics --out results/metrics.json

## 4. Results for the README ✓

In [ ]:
!python -m deploy.report --results results/loadtest.json --metrics results/metrics.json
from IPython.display import Image; Image('results/loadtest.png')

In [ ]:
# GPU memory in use (weights + KV cache reserved by --gpu-memory-utilization)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 5. (Optional) Use this endpoint from 01-rag and 02-agents
A Cloudflare quick tunnel gives you a public URL while the notebook runs. Then set `LLM_BASE_URL=<url>/v1` and `LLM_MODEL=unsloth/Llama-3.2-1B-Instruct` in the other projects.

⚠️ Anyone with the URL can use it while it's up. Stop the cell when you're done.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:8000 > tunnel.log 2>&1 &
import re, time; time.sleep(8)
print(re.findall(r'https://[\w-]+\.trycloudflare\.com', open('tunnel.log').read()))

## 6. Save results
Download these and commit them, along with the updated README.md.

In [ ]:
!zip -qr results.zip results README.md
from google.colab import files; files.download('results.zip')